# Métricas de Evaluación Aplicadas a Negocios
## F1, BLEU, ROUGE, Perplexity y BERTScore

**Objetivo:** Aprender a evaluar cuantitativamente la calidad de modelos de PLN
en contextos empresariales.

**Casos de negocio:**
1. Evaluar un sistema NER para extracción de datos de facturas
2. Medir calidad de resúmenes automáticos de reportes
3. Comparar modelos de generación de descripciones de productos
4. Seleccionar la mejor métrica según la tarea empresarial

---

## 1. Instalación de dependencias

In [4]:
# ============================================================
# INSTALACIÓN
# ============================================================

!pip install -q transformers evaluate sacrebleu rouge_score
!pip install -q bert-score scikit-learn pandas

print("Dependencias instaladas")

Dependencias instaladas


## 2. Caso de Negocio: F1-Score para evaluar NER en facturas

**Escenario:** Implementamos un sistema NER para extraer automáticamente datos
de facturas (proveedor, monto, fecha, NIT). ¿Qué tan preciso es?

---

In [5]:
# ============================================================
# CASO 1: F1-SCORE PARA NER EN FACTURAS COMERCIALES
# ============================================================

from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Simulamos etiquetas de un sistema NER aplicado a facturas
# Cada token tiene una etiqueta BIO (Beginning, Inside, Outside)

# Etiquetas reales (anotadas por humanos)
y_true = [
    "B-PROVEEDOR", "I-PROVEEDOR",      # "Cementos Argos"
    "O",                                 # "factura"
    "B-NIT",                             # "890.900.943-1"
    "O", "O",                            # "por valor"
    "B-MONTO", "I-MONTO",               # "$45.000.000"
    "O",                                 # "fecha"
    "B-FECHA", "I-FECHA", "I-FECHA",     # "15 de enero"
]

# Etiquetas predichas por nuestro modelo NER
y_pred = [
    "B-PROVEEDOR", "I-PROVEEDOR",      # ✅ Correcto
    "O",                                 # ✅ Correcto
    "B-NIT",                             # ✅ Correcto
    "O", "O",                            # ✅ Correcto
    "B-MONTO", "I-MONTO",               # ✅ Correcto
    "O",                                 # ✅ Correcto
    "B-FECHA", "I-FECHA", "O",           # ❌ Fallo: no detectó el último token de fecha
]

print("EVALUACIÓN DEL SISTEMA NER PARA FACTURAS")
print("=" * 60)

# Reporte completo por clase
print(classification_report(y_true, y_pred, zero_division=0))

# Métricas globales
precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"\n RESUMEN:")
print(f"   Precisión (weighted): {precision:.3f}")
print(f"   Recall (weighted):    {recall:.3f}")
print(f"   F1-Score (weighted):  {f1:.3f}")

# Interpretación para el negocio
print(f"\n INTERPRETACIÓN EMPRESARIAL:")
print(f"   El sistema detecta correctamente el {recall*100:.1f}% de las entidades reales.")
print(f"   De las entidades que detecta, el {precision*100:.1f}% son correctas.")
print(f"   El F1 de {f1:.3f} indica un {'buen' if f1 > 0.85 else 'regular'} equilibrio entre precisión y cobertura.")

EVALUACIÓN DEL SISTEMA NER PARA FACTURAS
              precision    recall  f1-score   support

     B-FECHA       1.00      1.00      1.00         1
     B-MONTO       1.00      1.00      1.00         1
       B-NIT       1.00      1.00      1.00         1
 B-PROVEEDOR       1.00      1.00      1.00         1
     I-FECHA       1.00      0.50      0.67         2
     I-MONTO       1.00      1.00      1.00         1
 I-PROVEEDOR       1.00      1.00      1.00         1
           O       0.80      1.00      0.89         4

    accuracy                           0.92        12
   macro avg       0.97      0.94      0.94        12
weighted avg       0.93      0.92      0.91        12


 RESUMEN:
   Precisión (weighted): 0.933
   Recall (weighted):    0.917
   F1-Score (weighted):  0.907

 INTERPRETACIÓN EMPRESARIAL:
   El sistema detecta correctamente el 91.7% de las entidades reales.
   De las entidades que detecta, el 93.3% son correctas.
   El F1 de 0.907 indica un buen equilibrio ent

In [6]:
# ============================================================
# CÁLCULO MANUAL DE F1 PARA ENTENDER LA FÓRMULA
# ============================================================

print("CÁLCULO MANUAL DE F1-SCORE")
print("=" * 50)

# Ejemplo simplificado: clasificación binaria
# Supongamos que nuestro modelo clasifica correos como "urgente" o "no urgente"

# Resultados del modelo:
TP = 45   # Urgentes correctamente detectados
FP = 5    # No urgentes clasificados como urgentes (falsa alarma)
FN = 10   # Urgentes no detectados (se le escaparon)
TN = 140  # No urgentes correctamente ignorados

# Fórmulas
precision = TP / (TP + FP)   # De los que marqué como urgentes, ¿cuántos lo eran?
recall = TP / (TP + FN)      # De los urgentes reales, ¿cuántos encontré?
f1 = 2 * (precision * recall) / (precision + recall)  # Media armónica

print(f"   TP={TP}, FP={FP}, FN={FN}, TN={TN}")
print(f"\n   Precisión = TP/(TP+FP) = {TP}/({TP}+{FP}) = {precision:.3f}")
print(f"   → 'De los correos marcados como urgentes, el {precision*100:.1f}% realmente lo eran'")
print(f"\n   Recall    = TP/(TP+FN) = {TP}/({TP}+{FN}) = {recall:.3f}")
print(f"   → 'Detectamos el {recall*100:.1f}% de todos los correos urgentes'")
print(f"\n   F1        = 2×(P×R)/(P+R) = {f1:.3f}")
print(f"   → 'Balance general del clasificador: {f1*100:.1f}%'")

CÁLCULO MANUAL DE F1-SCORE
   TP=45, FP=5, FN=10, TN=140

   Precisión = TP/(TP+FP) = 45/(45+5) = 0.900
   → 'De los correos marcados como urgentes, el 90.0% realmente lo eran'

   Recall    = TP/(TP+FN) = 45/(45+10) = 0.818
   → 'Detectamos el 81.8% de todos los correos urgentes'

   F1        = 2×(P×R)/(P+R) = 0.857
   → 'Balance general del clasificador: 85.7%'


## 3. Caso de Negocio: BLEU y ROUGE para resúmenes de reportes

**Escenario:** Comparamos dos modelos de resumen automático para decidir
cuál implementar en producción para generar resúmenes ejecutivos.

---

In [7]:
# ============================================================
# CASO 2: BLEU Y ROUGE PARA EVALUAR RESÚMENES EMPRESARIALES
# ============================================================

import evaluate

# Cargar métricas
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")

# Reporte original
reporte = """
The company achieved total revenue of 4.2 billion dollars in Q3 2024,
a 12 percent increase year over year. Cloud division grew 18 percent.
Operating expenses rose 8 percent to 2.8 billion. Net income was
890 million, beating expectations. A new partnership with Azure was
announced for Latin America expansion. Headcount grew by 2000 to 45000.
"""

# Resumen de referencia (escrito por un analista humano)
referencia = "Revenue reached $4.2B in Q3 2024, up 12%. Cloud grew 18%. Net income $890M beat expectations. Azure partnership for LatAm."

# Resúmenes generados por dos modelos diferentes
resumen_modelo_A = "The company reported 4.2 billion in revenue for Q3 2024, up 12 percent. Net income was 890 million."
resumen_modelo_B = "Revenue growth of 12 percent was reported. Cloud services expanded significantly. New partnerships were formed in Latin America."

print("EVALUACIÓN DE MODELOS DE RESUMEN")
print("=" * 70)
print(f"\nReferencia humana:")
print(f"   {referencia}")

for nombre, resumen in [("Modelo A", resumen_modelo_A), ("Modelo B", resumen_modelo_B)]:
    # BLEU: mide coincidencia de n-gramas (orientado a precisión)
    bleu_result = bleu_metric.compute(
        predictions=[resumen],
        references=[[referencia]]
    )

    # ROUGE: mide solapamiento de contenido (orientado a recall)
    rouge_result = rouge_metric.compute(
        predictions=[resumen],
        references=[referencia]
    )

    print(f"\n📝 {nombre}: {resumen}")
    print(f"   BLEU:    {bleu_result['score']:6.2f}")
    print(f"   ROUGE-1: {rouge_result['rouge1']:.4f}  (unigramas)")
    print(f"   ROUGE-2: {rouge_result['rouge2']:.4f}  (bigramas)")
    print(f"   ROUGE-L: {rouge_result['rougeL']:.4f}  (subsecuencia más larga)")

OSError: [WinError 1114] Error en una rutina de inicialización de biblioteca de vínculos dinámicos (DLL). Error loading "C:\Users\epuerta\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
# ============================================================
# COMPARACIÓN T5 vs GPT-2 CON MÉTRICAS
# ============================================================

from transformers import pipeline

# Generar resúmenes con ambos modelos
summarizer = pipeline("summarization", model="t5-small", tokenizer="t5-small")
gen = pipeline("text-generation", model="gpt2")

# T5: resumen controlado
t5_result = summarizer(reporte, max_length=40, min_length=10, do_sample=False)
texto_t5 = t5_result[0]["summary_text"]

# GPT-2: continuación libre (luego truncamos)
gpt2_result = gen(
    reporte.strip()[:80],
    max_length=120,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    pad_token_id=50256  # eos_token_id de GPT-2
)
texto_gpt2 = gpt2_result[0]["generated_text"][:150]

print("⚔️ T5 vs GPT-2: EVALUACIÓN CON MÉTRICAS")
print("=" * 70)

for nombre, texto_gen in [("T5", texto_t5), ("GPT-2", texto_gpt2)]:
    bleu_r = bleu_metric.compute(predictions=[texto_gen], references=[[referencia]])
    rouge_r = rouge_metric.compute(predictions=[texto_gen], references=[referencia])

    print(f"\n {nombre}:")
    print(f"   Texto: {texto_gen[:100]}...")
    print(f"   BLEU={bleu_r['score']:5.1f} | ROUGE-1={rouge_r['rouge1']:.3f} | ROUGE-L={rouge_r['rougeL']:.3f}")

print("\n💡 T5 debería tener mejores scores porque fue diseñado para resumen.")
print("   GPT-2 genera texto libre, no optimizado para preservar la referencia.")

## 4. Caso de Negocio 3: Perplexity para evaluar calidad de texto

**Escenario:** Queremos medir qué tan fluidos son los textos generados
por nuestro chatbot de servicio al cliente.

---

In [ ]:
# ============================================================
# CASO 3: PERPLEXITY PARA EVALUAR FLUIDEZ DE CHATBOT
# ============================================================

import torch
import math
from transformers import AutoTokenizer, AutoModelForCausalLM

# Cargar GPT-2 para calcular perplexity
tok = AutoTokenizer.from_pretrained("gpt2")
mod = AutoModelForCausalLM.from_pretrained("gpt2")
mod.eval()

def calcular_perplexity(texto):
    """
    Calcula la perplexity de un texto usando GPT-2.
    Menor perplexity = texto más fluido/natural según el modelo.
    """
    encodings = tok(texto, return_tensors="pt")
    with torch.no_grad():
        outputs = mod(**encodings, labels=encodings["input_ids"])
    return math.exp(outputs.loss.item())

# Respuestas simuladas de un chatbot empresarial
respuestas_chatbot = {
    "Fluida y profesional": "Thank you for contacting us. I understand your concern about the delayed shipment. Let me check the status right away.",
    "Aceptable pero robótica": "Your order number 4521 is being processed. Delivery expected in 3 to 5 business days. Contact support for more info.",
    "Incoherente (mal modelo)": "Order shipment yes thank for your the patience delivery soon maybe contact.",
    "Repetitiva (problema de generación)": "We are sorry. We are sorry. We are sorry for the inconvenience. We are sorry.",
}

print("🎧 EVALUACIÓN DE FLUIDEZ DEL CHATBOT (PERPLEXITY)")
print("=" * 70)
print(f"{'Tipo de respuesta':35s} | {'Perplexity':>12s} | {'Calidad':>10s}")
print("-" * 65)

for tipo, texto in respuestas_chatbot.items():
    ppl = calcular_perplexity(texto)
    calidad = "🟢 Buena" if ppl < 80 else "🟡 Regular" if ppl < 200 else "🔴 Mala"
    print(f"{tipo:35s} | {ppl:12.1f} | {calidad}")

print("\n💡 INTERPRETACIÓN:")
print("   Perplexity BAJA (<100) → El texto es fluido y natural")
print("   Perplexity ALTA (>200) → El texto es incoherente o antinatural")
print("   ⚠️ Perplexity NO mide si el texto es verdadero o relevante")

## 5. BERTScore: Evaluación semántica avanzada

**Escenario:** BLEU penaliza paráfrasis (reformulaciones con el mismo significado).
BERTScore usa embeddings para capturar similitud semántica.

---

In [ ]:
# ============================================================
# CASO 4: BERTSCORE vs BLEU - CAPTURANDO SEMÁNTICA
# ============================================================

from bert_score import score as bert_score
import evaluate

bleu_metric = evaluate.load("sacrebleu")

# Referencia: descripción de producto escrita por un humano
referencia = "High-quality wireless Bluetooth headphones with active noise cancellation and 30-hour battery life."

# Diferentes versiones generadas
candidatos = {
    "Exacta":      "High-quality wireless Bluetooth headphones with active noise cancellation and 30-hour battery life.",
    "Paráfrasis":  "Premium wireless earphones featuring ANC technology and a battery that lasts 30 hours.",
    "Parcial":     "Wireless headphones with noise cancellation.",
    "Irrelevante": "The company reported strong quarterly earnings with revenue growth.",
}

print("🎧 BERTSCORE vs BLEU: EVALUANDO DESCRIPCIONES DE PRODUCTO")
print("=" * 70)

refs_list = [referencia] * len(candidatos)
cands_list = list(candidatos.values())

# Calcular BERTScore para todos los candidatos a la vez
P, R, F1 = bert_score(cands_list, refs_list, lang="en", verbose=False)

print(f"\n Referencia: {referencia}")
print(f"\n{'Tipo':15s} | {'BLEU':>8s} | {'BERTScore':>10s} | Texto")
print("-" * 85)

for i, (tipo, cand) in enumerate(candidatos.items()):
    bleu_r = bleu_metric.compute(predictions=[cand], references=[[referencia]])
    print(f"{tipo:15s} | {bleu_r['score']:7.1f} | {F1[i].item():10.4f} | {cand[:55]}...")

print("\n💡 OBSERVACIONES CLAVE:")
print("   • La PARÁFRASIS tiene BLEU bajo pero BERTScore alto (reconoce el significado)")
print("   • BLEU penaliza reformulaciones aunque el significado sea idéntico")
print("   • BERTScore es más adecuado cuando hay variabilidad en la expresión")
print("   • Para e-commerce, BERTScore evalúa mejor las descripciones de productos")

## 6. Taller integrador: Evaluación completa de un pipeline empresarial

**Escenario:** Evaluamos un pipeline completo de generación de resúmenes
para reportes financieros usando TODAS las métricas aprendidas.

---

In [ ]:
# ============================================================
# TALLER: EVALUACIÓN COMPLETA DE PIPELINE EMPRESARIAL
# ============================================================

import evaluate
from transformers import pipeline
from bert_score import score as bert_score

# Configuración
bleu_m = evaluate.load("sacrebleu")
rouge_m = evaluate.load("rouge")
summarizer = pipeline("summarization", model="t5-small")

# Dataset de reportes empresariales con referencias humanas
dataset = [
    {
        "reporte": "Apple Inc reported revenue of $89.5 billion for Q4 2024, a 6% increase year over year. iPhone sales accounted for $46 billion. Services revenue hit a record $22.3 billion. The company announced a $90 billion share repurchase program.",
        "referencia": "Apple Q4 revenue was $89.5B, up 6%. iPhone $46B. Services record $22.3B. $90B buyback announced."
    },
    {
        "reporte": "Tesla delivered 435,000 vehicles in Q3 2024. Revenue reached $25.2 billion. The company reduced production costs by 8% and expanded Supercharger network to 50,000 stations globally. Cybertruck production ramped to 2,500 units per week.",
        "referencia": "Tesla delivered 435K vehicles, earned $25.2B. Cut costs 8%. 50K Superchargers. Cybertruck at 2.5K/week."
    },
    {
        "reporte": "Microsoft cloud revenue surpassed $34 billion in Q2 2024. Azure grew 29% driven by AI workloads. GitHub Copilot reached 1.8 million paid subscribers. The company invested $13 billion in OpenAI partnership expansion.",
        "referencia": "Microsoft cloud $34B. Azure up 29% on AI. Copilot 1.8M subscribers. $13B OpenAI investment."
    },
]

print("EVALUACIÓN COMPLETA DE PIPELINE DE RESÚMENES")
print("=" * 80)

# Recopilar todos los resultados
resultados = []
textos_gen = []
textos_ref = []

for i, item in enumerate(dataset):
    # Generar resumen con T5
    resumen = summarizer(item["reporte"], max_length=50, min_length=15, do_sample=False)
    texto_gen = resumen[0]["summary_text"]
    ref = item["referencia"]

    textos_gen.append(texto_gen)
    textos_ref.append(ref)

    # Calcular BLEU
    bleu_r = bleu_m.compute(predictions=[texto_gen], references=[[ref]])

    # Calcular ROUGE
    rouge_r = rouge_m.compute(predictions=[texto_gen], references=[ref])

    resultados.append({
        "reporte": i+1,
        "bleu": bleu_r["score"],
        "rouge1": rouge_r["rouge1"],
        "rouge2": rouge_r["rouge2"],
        "rougeL": rouge_r["rougeL"],
    })

    print(f"\n📄 Reporte {i+1}:")
    print(f"   Original:  {item['reporte'][:80]}...")
    print(f"   Referencia: {ref}")
    print(f"   T5:         {texto_gen}")
    print(f"   BLEU={bleu_r['score']:5.1f} | R1={rouge_r['rouge1']:.3f} | R2={rouge_r['rouge2']:.3f} | RL={rouge_r['rougeL']:.3f}")

# Calcular BERTScore en batch
P, R, F1_bert = bert_score(textos_gen, textos_ref, lang="en", verbose=False)
for i in range(len(resultados)):
    resultados[i]["bertscore"] = F1_bert[i].item()

# Resumen final
import pandas as pd
df_resultados = pd.DataFrame(resultados)
print("\n\n📋 TABLA RESUMEN DE MÉTRICAS:")
print(df_resultados.to_string(index=False, float_format="%.3f"))
print(f"\n📈 PROMEDIOS:")
print(f"   BLEU:      {df_resultados['bleu'].mean():.2f}")
print(f"   ROUGE-1:   {df_resultados['rouge1'].mean():.3f}")
print(f"   ROUGE-L:   {df_resultados['rougeL'].mean():.3f}")
print(f"   BERTScore: {df_resultados['bertscore'].mean():.3f}")

## 7. Guía de selección de métricas por tarea empresarial

---

In [ ]:
# ============================================================
# GUÍA DE SELECCIÓN DE MÉTRICAS POR TAREA
# ============================================================

import pandas as pd

guia = pd.DataFrame([
    {"Tarea empresarial": "Clasificación de correos",     "Métrica principal": "F1-Score",    "Complemento": "Matriz de confusión", "Razón": "Equilibrio entre falsos positivos y negativos"},
    {"Tarea empresarial": "NER en facturas/contratos",    "Métrica principal": "F1 (seqeval)","Complemento": "Precisión por entidad","Razón": "Evalúa entidades completas, no solo tokens"},
    {"Tarea empresarial": "Traducción de documentos",     "Métrica principal": "BLEU",        "Complemento": "BERTScore",           "Razón": "N-gramas + similitud semántica"},
    {"Tarea empresarial": "Resumen de reportes",          "Métrica principal": "ROUGE",       "Complemento": "BERTScore + humana",  "Razón": "Cobertura de ideas clave + semántica"},
    {"Tarea empresarial": "Chatbot de servicio",          "Métrica principal": "Perplexity",  "Complemento": "Evaluación humana",   "Razón": "Fluidez + relevancia de respuestas"},
    {"Tarea empresarial": "Descripciones de productos",   "Métrica principal": "BERTScore",   "Complemento": "ROUGE + humana",      "Razón": "Las paráfrasis son válidas y frecuentes"},
    {"Tarea empresarial": "Detección de sentimiento",     "Métrica principal": "F1-Score",    "Complemento": "AUC-ROC",             "Razón": "Clases desbalanceadas (más neutral que negativo)"},
])

print("📋 GUÍA DE SELECCIÓN DE MÉTRICAS POR TAREA EMPRESARIAL")
print("=" * 100)
print(guia.to_string(index=False))

print("\n\n⚠️ REGLA DE ORO:")
print("   Nunca confíes en una sola métrica.")
print("   Combina métricas automáticas + evaluación humana.")
print("   El contexto del negocio determina qué métrica importa más.")

## 8. Resumen y conclusiones de la Sesión 3

### Lo que aprendimos:
- **F1-Score:** Ideal para clasificación y NER. Equilibra precisión y recall.
- **BLEU:** Para traducción y textos con referencia exacta. Orientado a precisión de n-gramas.
- **ROUGE:** Para resumen. Mide cobertura de contenido clave.
- **Perplexity:** Para fluidez de modelos generativos. No mide veracidad.
- **BERTScore:** Para similitud semántica. Reconoce paráfrasis que BLEU penaliza.

### Aplicaciones de negocio cubiertas:
1. ✅ Evaluación de NER en facturas comerciales
2. ✅ Comparación de modelos de resumen para reportes financieros
3. ✅ Evaluación de fluidez de chatbot empresarial
4. ✅ BERTScore vs BLEU para descripciones de productos
5. ✅ Pipeline completo de evaluación multi-métrica
6. ✅ Guía de selección de métricas por tarea empresarial

---
*Fin de la Semana 3: Procesamiento Avanzado y Evaluación en PLN*
